In [2]:
%run ../../code/imports.ipynb

Set LOG_LEVEL="INFO" before running the import file to get moar output.
Set LOG_FORMAT to change log format.
IPython: 8.5.0
Numpy (np): 1.23.5
Scipy (sp, stats): 1.15.2
Pandas (pd): 2.2.3
MatPlotLib (mpl, plt): 3.10.1
Seaborn (sns): 0.13.2
Scikit-Learn (sk): 1.1.2
Statsmodels (sm): 0.14.2
Patsy (pt): 0.5.6
SQLAlchemy (sa): 2.0.35
Gensim: 4.3.3
ROOT_DIR: 5150_Analytics
CODE_DIR: 5150_Analytics/code
DATA_DIR: 5150_Analytics/data
FIG_DIR: 5150_Analytics/labs_hw/week12_AI/figures


In [3]:
from pyedgar import EDGARIndex, Filing
from bs4 import BeautifulSoup

In [4]:
df = EDGARIndex()['10-K'].query('filedate.dt.year == 2025')
df.head(1)

,cik,name,form,filedate,accession
45,1750,AAR CORP,10-K,2025-07-22,0001410578-25-001475


In [21]:
SIC_KEY = "assigned-sic"

In [6]:
REVENUE_TAGS = [
    "us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax",
    "us-gaap:Revenues",
    "us-gaap:RevenueFromContractWithCustomerIncludingAssessedTax",
    "us-gaap:SalesRevenueNet",
    "us-gaap:SalesRevenueGoodsNet",
    "us-gaap:SalesRevenueServicesNet",
    "us-gaap:RegulatedAndUnregulatedOperatingRevenue",
    "us-gaap:InterestAndDividendIncomeOperating",
    "us-gaap:RealEstateRevenueNet",
]

def extract_latest_revenue(soup: BeautifulSoup) -> dict:
    """
    Extracts only the most recent period's consolidated revenue from an iXBRL soup object.
    """
    # 1. Identify non-dimensional context IDs
    valid_contexts = {}
    for ctx in soup.find_all(re.compile(r"(xbrli:)?context$", re.I)):
        if not ctx.find(re.compile(r"(xbrli:)?segment$|(xbrldi:)?explicitMember$", re.I)):
            ctx_id = ctx.get("id")
            period = ctx.find(re.compile(r"(xbrli:)?endDate$", re.I))
            if ctx_id and period:
                valid_contexts[ctx_id] = period.get_text(strip=True)

    # 2. Iterate through the tag waterfall
    for tag_name in REVENUE_TAGS:
        facts = soup.find_all(
            re.compile(r"ix:nonFraction$", re.I),
            attrs={"name": re.compile(f"^{re.escape(tag_name)}$", re.I)}
        )
        
        results = {}
        for fact in facts:
            ctx_ref = fact.get("contextref")
            if ctx_ref not in valid_contexts:
                continue
            
            raw_val = fact.get_text(strip=True).replace(",", "")
            if not raw_val or raw_val in ("-", "—"):
                continue
                
            try:
                val = float(raw_val)
                if fact.get("sign") == "-":
                    val = -val
                scale = int(fact.get("scale", 0))
                
                end_date = valid_contexts[ctx_ref]
                results[end_date] = val * (10 ** scale)
            except ValueError:
                continue
                
        # Return only the latest period end date (ISO YYYY-MM-DD strings sort chronologically)
        if results:
            latest_date = max(results.keys())
            return {
                "tag": tag_name,
                "period_end": latest_date,
                "revenue": results[latest_date]
            }

    return {"tag": None, "period_end": None, "revenue": None}

In [7]:
_res = dict()

In [8]:
import os
from concurrent.futures import ProcessPoolExecutor

# Processes (fork on Linux) sidestep the GIL and any non-thread-safe state in pyedgar.
MAX_WORKERS = min(os.cpu_count() or 1, 16)


def process_filing(row: dict) -> dict:
    """Pull SIC, tickers, and latest revenue for one filing. Never raises."""
    out = {**row, 'filedate': f"{row['filedate']:%Y-%m-%d}"}
    soup = None
    try:
        filing = Filing(**row)
        out['sic'] = filing.headers.get(SIC_KEY)
    except Exception:
        pass
    try:
        soup = BeautifulSoup(filing.documents[0]['full_text'], "lxml")
        out['tickers'] = sorted(
            (tag.get_text(strip=True) for tag in soup.find_all(attrs={"name": "dei:TradingSymbol"})),
            key=len,
        )
    except Exception:
        pass
    try:
        out['revenue'] = extract_latest_revenue(soup)['revenue']
    except Exception:
        pass
    return out

In [9]:
# Resumable: skips accessions already in _res.
todo = [row.to_dict() for _, row in df.iterrows() if row.accession not in _res]

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    for result in tqdm(ex.map(process_filing, todo, chunksize=16), total=len(todo)):
        _res[result['accession']] = result

  0%|          | 0/7419 [00:00<?, ?it/s]

In [36]:
dfout = (
    pd.DataFrame.from_dict(_res, orient='index')
    .reset_index(drop=True)
    .assign(
        ticker=lambda df: df.tickers.apply(lambda x: x[0] if isinstance(x, list) and x else None),
        sic=lambda df: df.sic.apply(lambda x: x[0] if isinstance(x, list) and x else x),
    )
)
for c in 'tickers sic_name'.split(): 
    if c in dfout: del dfout[c]
display(dfout.describe(include='all').fillna(''))

dfout = (
    dfout
    .dropna(how='any')
    .query('form=="10-K"')
    .drop_duplicates('cik')
    .drop_duplicates('ticker')
    .drop(columns='form filedate'.split())
)
display(dfout.describe(include='all').fillna(''))

dfout.head(1)

,cik,name,form,filedate,accession,sic,revenue,ticker
count,"7,155",7155,7155,7155,7155,6667,"4,683",5123
unique,,6411,4,246,7155,393,,4566
top,,BrightRock Gold Corp,10-K,2025-03-31,0001410578-25-001475,6189,,None
freq,,6,6431,474,1,750,,5
mean,"1,383,492",,,,,,"4,862,766,014",
std,"527,857",,,,,,"24,899,767,161",
min,"1,750",,,,,,"-7,943,000",
25%,"1,056,286",,,,,,"19,342,404",
50%,"1,543,637",,,,,,"304,326,000",
75%,"1,798,244",,,,,,"2,086,594,500",


,cik,name,accession,sic,revenue,ticker
count,"3,772",3772,3772,3772,"3,772",3772
unique,,3772,3772,380,,3772
top,,AAR CORP,0001410578-25-001475,2834,,AIR
freq,,1,1,283,,1
mean,"1,219,395",,,,"5,693,903,710",
std,"553,303",,,,"27,307,044,455",
min,"1,750",,,,"-7,943,000",
25%,"890,522",,,,"56,926,250",
50%,"1,352,810",,,,"539,719,500",
75%,"1,672,182",,,,"2,735,179,250",


,cik,name,accession,sic,revenue,ticker
0,1750,AAR CORP,0001410578-25-001475,3720,"2,780,500,000",AIR


In [37]:
dfout.to_csv('comp-dashboard-starter/sec_companies.csv', index=False)